## Data Encoding Notes

### 1. Nominal Encoding (One-Hot Encoding - OHE)

#### Definition
Used for **categorical data with no order** (nominal categories).

#### Idea
Create **binary columns (0/1)** for each category.

#### Example
Color = [Red, Blue, Green]

→ One-Hot:
- Red → [1, 0, 0]  
- Blue → [0, 1, 0]  
- Green → [0, 0, 1]

#### Pros
- No false ordering introduced  
- Works well for most ML models  

#### Cons
- High dimensionality (curse of dimensionality)  
- Inefficient for many categories  


In [1]:
import pandas as pd 
from sklearn.preprocessing import OneHotEncoder as ohe

In [2]:
## create a simple dataframe
df = pd.DataFrame({
    'color':['red','blue','green','yellow','red','blue','green','orange','pink']
})
df.head()

,color
0,red
1,blue
2,green
3,yellow
4,red


In [10]:
## create an instance of OneHotEncoder
encoder=ohe()
## perform fit and transform
encoded=encoder.fit_transform(df[['color']]).toarray()
encoded

array([[0., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.]])

In [9]:
import pandas as pd 
encoder_df=pd.DataFrame(encoded,columns=encoder.get_feature_names_out())
encoder_df

,color_blue,color_green,color_orange,color_pink,color_red,color_yellow
0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0
6,0.0,1.0,0.0,0.0,0.0,0.0
7,0.0,0.0,1.0,0.0,0.0,0.0
8,0.0,0.0,0.0,1.0,0.0,0.0



### 2. Label Encoding & Ordinal Encoding

#### Label Encoding

#### Definition
Assigns **unique integer to each category**

#### Example
Color:
- Red → 0  
- Blue → 1  
- Green → 2  

#### Issue
- Introduces false order (model may think Green > Blue)


In [11]:
df.head()

,color
0,red
1,blue
2,green
3,yellow
4,red


In [12]:
from sklearn.preprocessing import LabelEncoder as le 
lbl_encoder=le()

In [14]:
lbl_encoder.fit_transform(df['color'])

array([4, 0, 1, 5, 4, 0, 1, 2, 3])

In [15]:
lbl_encoder.transform(['red'])

array([4])

In [16]:
lbl_encoder.transform(['blue'])

array([0])

In [17]:
lbl_encoder.transform(['pink'])

array([3])

#### Ordinal Encoding

##### Definition
Used when **categories have a meaningful order**

##### Example
Size:
- Small → 0  
- Medium → 1  
- Large → 2  

#### Pros
- Preserves order information  
- Memory efficient  

#### Cons
- Not suitable for nominal data 


In [22]:
from sklearn.preprocessing import OrdinalEncoder

In [23]:
df = pd.DataFrame({
    'size':['small','medium','large','medium','small','large']
})
df.head()

,size
0,small
1,medium
2,large
3,medium
4,small


In [24]:
ord_encoder=OrdinalEncoder(categories=[['small','medium','large']])
ord_encoder.fit_transform(df[['size']])


array([[0.],
       [1.],
       [2.],
       [1.],
       [0.],
       [2.]])

In [27]:
ord_encoder.transform([['medium']])

d:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


array([[1.]])

### 3. Target Guided Ordinal Encoding

#### Definition
Categories are encoded based on **target variable statistics** (usually mean).

#### Idea
Replace each category with:
Encoding = mean(target | category)

#### Example
Category vs Target Mean:
- A → 0.8  
- B → 0.5  
- C → 0.2  

Encoded:
- A → 0.8  
- B → 0.5  
- C → 0.2  

#### Pros
- Captures relationship with target  
- Reduces dimensionality  

#### Cons
- Data leakage risk (use CV / train-only stats)  
- Sensitive to noise  


In [29]:
import pandas as pd
# create a sample dataframe with a cotegorical variable and a target variable
df = pd.DataFrame({
    'city' : ['New York', 'Los Angeles', 'Chicago', 'New York', 'Phoenix'],
    'price' : [120, 100, 90, 110, 90]
})
df

,city,price
0,New York,120
1,Los Angeles,100
2,Chicago,90
3,New York,110
4,Phoenix,90


In [32]:
mean_price=df.groupby('city')['price'].mean().to_dict()

In [35]:
df['city_encoded']=df['city'].map(mean_price)
df

,city,price,city_encoded
0,New York,120,115.0
1,Los Angeles,100,100.0
2,Chicago,90,90.0
3,New York,110,115.0
4,Phoenix,90,90.0



### Summary Table

| Encoding Type | Use Case | Order Preserved | Dimensionality |
|--------------|---------|----------------|----------------|
| One-Hot      | Nominal | No             | High           |
| Label        | Nominal (not ideal) | No (false order) | Low |
| Ordinal      | Ordered categories | Yes | Low |
| Target Guided| Target-based | Yes (data-driven) | Low |

In [37]:
## a practice using a dataset
import seaborn as sns
df=sns.load_dataset('tips')
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [39]:
## changing time based on total bill
df['time_encoded']=df['total_bill'].apply(lambda x: 'lunch' if x<20 else 'dinner')


In [41]:
df[['total_bill','time','time_encoded']]

,total_bill,time,time_encoded
0,16.99,Dinner,lunch
1,10.34,Dinner,lunch
2,21.01,Dinner,dinner
3,23.68,Dinner,dinner
4,24.59,Dinner,dinner
...,...,...,...
239,29.03,Dinner,dinner
240,27.18,Dinner,dinner
241,22.67,Dinner,dinner
242,17.82,Dinner,lunch
